In [2]:
%pip install selenium webdriver-manager beautifulsoup4
%pip install undetected-chromedriver
%pip install pandas


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

from bs4 import BeautifulSoup
import pandas as pd
import time
import requests
import os, csv, re

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# IMDb Reviews

In [3]:
movies = [
    "The Shawshank Redemption",
    "Parasite",
    "The Godfather",
    "Spider-Man: Into the Spider-Verse",
    "Raiders of the Lost Ark",
    "The Dark Knight",
    "12 Angry Men",
    "Pulp Fiction",
    "Harakiri",
    "Spirited Away",
    "Coco",
    "Goodfellas",
    "The Good, the Bad and the Ugly",
    "Toy Story 3",
    "Rear Window",
    "City Lights",
    "Alien",
    "The Silence of the Lambs",
    "Inception",
    "The Shining",
    "Se7en",
    "Casablanca",
    "One Flew Over the Cuckoo's Nest",
    "Seven Samurai",
    "It's a Wonderful Life",
    "Memento",
    "Psycho",
    "Singin' in the Rain",
    "City of God",
    "Apocalypse Now",
    "Whiplash",
    "Vertigo",
    "Your Name",
    "Citizen Kane",
    "The Pianist",
    "2001: A Space Odyssey",
    "Dr. Strangelove",
    "Double Indemnity",
    "Lawrence of Arabia",
    "Paths of Glory",
    "Toy Story",
    "Anatomy of a Fall",
    "Grave of the Fireflies",
    "North by Northwest",
    "Once Upon a Time in the West",
    "Amadeus",
    "Wall-E",
    "To Kill a Mockingbird",
    "Bicycle Thieves",
    "Eternal Sunshine of the Spotless Mind",
]

In [4]:
# Returns imdb reviews url for a given movie name with spaces, e.g. "The Batman"
def get_imdb_reviews_url_api(movie_name):
    # Use IMDB's suggestion API — fast, no Selenium needed
    search_query = movie_name.replace(" ", "_")
    api_url = f"https://v3.sg.media-imdb.com/suggestion/x/{search_query}.json"
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    response = requests.get(api_url, headers=headers)
    data = response.json()
    
    # Filter for movies only (qid == "movie") and grab the first one
    movies = [r for r in data.get("d", []) if r.get("qid") == "movie"]
    
    if not movies:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    movie_id = movies[0]["id"]  # e.g. tt6751668
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    
    print(f"{movie_name} url: {reviews_url}")
    return reviews_url

def scrape_reviews_imdb(driver, reviews_url):
    print("Scraping reviews...")
    driver.get(reviews_url)
    time.sleep(5)

    # Click "Load More" twice to scrape 75 reviews in total then truncate to 50
    for i in range(2):
        try:
            load_more = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more)
            time.sleep(3)
            print("Loaded more reviews...")
        except:
            print("All reviews loaded.")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    reviews = []
    articles = soup.find_all("article", class_="user-review-item")
    print(f"Found {len(articles)} reviews, parsing...")

    count = 1

    for article in articles:
        review = {}

        review["id"] = f"{count}"
        count += 1

        # Rating — only present if user gave one
        rating_tag = article.find("span", class_="ipc-rating-star--rating")
        review["user_rating"] = rating_tag.get_text(strip=True) if rating_tag else "N/A"

        # Title
        # title_tag = article.find("h3", class_="ipc-title__text")
        # review["review_title"] = title_tag.get_text(strip=True) if title_tag else "N/A"

        # Review text
        text_tag = article.find("div", attrs={"data-testid": "review-overflow"})
        if not text_tag:
            text_tag = article.find("div", class_=lambda c: c and "content" in c.lower())
        review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

        # Author
        # author_tag = article.find("a", attrs={"data-testid": "author-link"})
        # review["author"] = author_tag.get_text(strip=True) if author_tag else "N/A"

        # Date
        # date_tag = article.find("li", attrs={"data-testid": "review-date"})
        # review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

        reviews.append(review)

    reviews = reviews[:50]
    return reviews


In [ ]:
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

movie = "Parasite"
movie_url = get_imdb_reviews_url_api(movie)
reviews = scrape_reviews_imdb(driver, movie_url)

os.makedirs("reviews", exist_ok=True)
safe_name = re.sub(r'[^\w\s-]', '', movie).strip().replace(' ', '_')
filepath = f"reviews/{safe_name}_imdb_reviews.csv"

with open(filepath, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["id", "user_rating", "review_text"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(reviews)
    print(f"Saved to: {filepath}")

driver.quit()


Parasite url: https://www.imdb.com/title/tt6751668/reviews/
Scraping reviews...
Loaded more reviews...
Loaded more reviews...
Found 74 reviews, parsing...
Saved to: reviews/Parasite_imdb_reviews.csv


In [7]:
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

for movie in movies:
    movie_url = get_imdb_reviews_url_api(movie)
    reviews = scrape_reviews_imdb(driver, movie_url)

    filepath = f"{movie.replace(' ', '_')}_imdb_reviews.csv"

    os.makedirs("reviews", exist_ok=True)
    safe_name = re.sub(r'[^\w\s-]', '', movie).strip().replace(' ', '_')
    filepath = f"reviews/{safe_name}_imdb_reviews.csv"

    if os.path.exists(filepath):
        print(f"Skipping {movie}, already scraped.")
        continue

    with open(filepath, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["user_rating", "review_text"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(reviews)
        print(f"Saved to: {filepath}")

driver.quit()


The Shawshank Redemption url: https://www.imdb.com/title/tt0111161/reviews/
Scraping reviews...


KeyboardInterrupt: 

# Movie Metadata

In [ ]:
def get_movie_info_imdb(driver, main_url):
    print("Scraping movie info...")
    driver.get(main_url)
    time.sleep(5)

    print(f"Current Page Title: {driver.title}")
    
    if "Bot Check" in driver.title or "Just a moment" in driver.title:
        print("CRITICAL: IMDb is blocking the bot. Try turning off --headless.")
        
    soup = BeautifulSoup(driver.page_source, "html.parser")

    info = {}

    # Title
    tag = soup.find(attrs={"data-testid": "hero__primary-text"})
    info["title"] = tag.get_text(strip=True) if tag else "N/A"

    # Rating
    tag = soup.find(attrs={"data-testid": "hero-rating-bar__aggregate-rating__score"})
    info["imdb_rating"] = tag.get_text(strip=True).replace("/10", "") if tag else "N/A"

    # Year
    tag = soup.find(attrs={"data-testid": "hero-parent"})
    if tag:
        match = re.search(r'\b(19|20)\d{2}\b', tag.get_text())
        info["year"] = match.group(0) if match else "N/A"
    else:
        info["year"] = "N/A"

    # Budget
    tag = soup.find(attrs={"data-testid": "title-boxoffice-budget"})
    if tag:
        text = tag.get_text(strip=True).replace("Budget", "").replace("(estimated)", "").strip()
        info["budget"] = text
    else:
        info["budget"] = "N/A"

    # Gross US & Canada
    tag = soup.find(attrs={"data-testid": "title-boxoffice-grossdomestic"})
    if tag:
        text = tag.get_text(strip=True).replace("Gross US & Canada", "").strip()
        info["gross_us"] = text
    else:
        info["gross_us"] = "N/A"

    print(f"Movie info: {info}")
    return info